# Experiment Tracking: Vertex AI Experiments (MLflow)

What exactly did we do to produce those results, and can we reproduce/compare them later?

It's the system for recording:
- experiment
- run
- parameters
- metrics
- comparisons

Concept:
Model training > parameters + metrics + artifacts > Experiment Tracking > historical experiment record

| MLflow | Vertex AI equivalent |
|---|---|
| `mlflow.set_experiment(name)` | `aiplatform.init(experiment=name)` |
| `mlflow.start_run()` | `aiplatform.start_run(run="...")` |
| `mlflow.log_param(k, v)` | `aiplatform.log_params({...})` |
| `mlflow.log_metric(k, v)` | `aiplatform.log_metrics({...})` |
| `mlflow.end_run()` | `aiplatform.end_run()` |
| `mlflow.search_runs()` / tracking UI | `aiplatform.get_experiment_df(name)` |
| MLflow Model Registry | Vertex AI Model Registry


### Project setup

In [15]:
from google.cloud import aiplatform
import pandas as pd

project = !gcloud config get-value project
PROJECT_ID = project[0]
BUCKET = PROJECT_ID
REGION = 'us-central1'

DATANAME = "synthetic_fashion_demand"
EXPERIMENT_NAME = "synthetic-fashion-demand-comparisons"   # Vertex AI experiment names: lowercase letters, numbers, dashes

aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)
print(f"Tracking to experiment: {EXPERIMENT_NAME}")

Tracking to experiment: synthetic-fashion-demand-comparisons


## Read from Model Comparison step

In [16]:
RESULTS_PREFIX = f"gs://{BUCKET}/experiments/model_comparison/latest"

results_log_df = pd.read_csv(f"{RESULTS_PREFIX}/results_log.csv")   # automl-only: rmse/mae/mape per context_window
overall_df = pd.read_csv(f"{RESULTS_PREFIX}/overall.csv")           # every algorithm: wape, n_skus

print(f"{len(overall_df)} runs to log")
overall_df.sort_values("wape")

12 runs to log


,algorithm,context_window,wape,n_skus
0,automl,8,0.281416,300.0
1,automl,52,0.312435,300.0
2,automl,0,0.357978,300.0
3,nhits,52,0.377740,278.0
4,nbeats,52,0.400921,278.0
5,nbeats,8,0.405870,300.0
6,nhits,104,0.409160,256.0
7,nhits,8,0.410423,300.0
8,nbeats,104,0.412987,256.0
9,nhits,26,0.428558,297.0


## Log every run

One Experiment Run per `(algorithm, context_window)` combination —
params describe *how* it was trained, metrics describe *how well*.

In [17]:
def log_experiment_run(row, extra_metrics=None):
    run_name = f"{row['algorithm']}-cw{row['context_window']}".replace("_", "-").lower()

    aiplatform.start_run(run=run_name)
    aiplatform.log_params({
        "algorithm": row["algorithm"],
        "context_window": str(row["context_window"]),   # "auto" for bqml — log_params wants consistent types, keep it a string
        "forecast_horizon": FORECAST_HORIZON,
        "seasonal_period": SEASONAL_PERIOD,
        "dataset": DATANAME,
    })
    metrics = {"wape": float(row["wape"]), "n_skus": int(row["n_skus"])}
    if extra_metrics:
        metrics.update({k: float(v) for k, v in extra_metrics.items() if pd.notna(v)})
    aiplatform.log_metrics(metrics)
    aiplatform.end_run()

FORECAST_HORIZON = 8
SEASONAL_PERIOD = 52

# automl rows carry rmse/mae/mape
automl_extra = results_log_df.set_index(["algorithm", "context_window"])[["rmse", "mae", "mape"]]

for _, row in overall_df.iterrows():
    key = (row["algorithm"], row["context_window"])
    extra = automl_extra.loc[key].to_dict() if key in automl_extra.index else None
    log_experiment_run(row, extra)

print(f"Logged {len(overall_df)} runs to experiment '{EXPERIMENT_NAME}'")

/var/tmp/ipykernel_9946/893250257.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  extra = automl_extra.loc[key].to_dict() if key in automl_extra.index else None


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-automl-cw8 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-automl-cw52 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-automl-cw0 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-nhits-cw52 to Experiment: synthetic-fashion-demand-comparisons


...

Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-nbeats-cw104 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-nhits-cw26 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-nbeats-cw26 to Experiment: synthetic-fashion-demand-comparisons


Associating projects/456832640267/locations/us-central1/metadataStores/default/contexts/synthetic-fashion-demand-comparisons-local-auto-arima-cwauto to Experiment: synthetic-fashion-demand-comparisons


Logged 12 runs to experiment 'synthetic-fashion-demand-comparisons'


## mlflow search_runs() equivalent

In [8]:
experiment_df = aiplatform.get_experiment_df(EXPERIMENT_NAME)
leaderboard = experiment_df.sort_values("metric.wape")[
    ["run_name", "param.algorithm", "param.context_window", "metric.wape", "metric.n_skus"]
]
leaderboard

,run_name,param.algorithm,param.context_window,metric.wape,metric.n_skus
11,automl-cw8,automl,8,0.281416,300.0
10,automl-cw52,automl,52,0.312435,300.0
9,automl-cw0,automl,0,0.357978,300.0
8,nhits-cw52,nhits,52,0.377740,278.0
7,nbeats-cw52,nbeats,52,0.400921,278.0
6,nbeats-cw8,nbeats,8,0.405870,300.0
5,nhits-cw104,nhits,104,0.409160,256.0
4,nhits-cw8,nhits,8,0.410423,300.0
3,nbeats-cw104,nbeats,104,0.412987,256.0
2,nhits-cw26,nhits,26,0.428558,297.0


## About "the champion"

The winning **run** just tells you *which* algorithm/context_window to deploy. 

- **automl** — already a Vertex Model resource 
- **bqml_arima_plus** — a BigQuery ML model. 
- **nbeats / nhits** — a darts model object, not a GCP resource yet.

## get_champion_metrics component

In [9]:
from kfp.dsl import Output, Metrics

# @dsl.component(base_image="python:3.11", packages_to_install=["google-cloud-aiplatform==1.60.0"])
def get_champion_metrics(
    experiment_name: str,
    champion_run_name: str,        # set by register_and_deploy when it promotes a run
    champion_metrics: Output[Metrics],
):
    from google.cloud import aiplatform

    aiplatform.init(project=PROJECT_ID, location=REGION)
    experiment_df = aiplatform.get_experiment_df(experiment_name)
    row = experiment_df[experiment_df["run_name"] == champion_run_name].iloc[0]

    champion_metrics.log_metric("wape", float(row["metric.wape"]))
    if "metric.mae" in row and pd.notna(row["metric.mae"]):
        champion_metrics.log_metric("mae", float(row["metric.mae"]))

In [13]:
print("CHAMPION")
print(row)

CHAMPION
algorithm           automl
context_window           8
wape              0.281416
n_skus               300.0
Name: 0, dtype: object
